# Backup Detection: Circuit Redundancy Analysis

This notebook demonstrates IRTK's `backup_detection` module:

1. **Detect backup heads** – find heads compensating for ablated heads
2. **Knockout compensation** – measure model compensation after ablation
3. **Circuit redundancy map** – pairwise redundancy relationships
4. **Critical vs backup** – classify heads by importance type
5. **Ablation recovery curve** – track metric as heads are restored

## Why This Matters

Backup head detection identifies redundant circuits that compensate when primary components are ablated. Understanding circuit redundancy is crucial for evaluating ablation results: if a 'backup' activates after knockout, the original ablation may underestimate the component's normal role.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.backup_detection import (
    detect_backup_heads, knockout_compensation,
    circuit_redundancy_map, critical_vs_backup,
    ablation_recovery_curve,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
backup = detect_backup_heads(model, tokens, metric_fn, target_layer=0, target_head=0)
print(f"Clean metric: {backup['clean_metric']:.4f}")
print(f"Ablated metric: {backup['ablated_metric']:.4f}")
print(f"Backup heads: {backup['backup_heads']}")

In [ ]:
comp = knockout_compensation(model, tokens, metric_fn)
print(f"Per-head ablation effects:\n{comp['per_head_ablation_effect'].round(4)}")
print(f"Most compensated: head {comp['most_compensated']}")

In [ ]:
redund = circuit_redundancy_map(model, tokens, metric_fn)
print(f"Redundancy matrix shape: {redund['redundancy_matrix'].shape}")
print(f"Most redundant pair: {redund['most_redundant_pair']}")
print(f"Redundancy score: {redund['redundancy_score']:.4f}")

In [ ]:
classify = critical_vs_backup(model, tokens, metric_fn)
print(f"Critical heads: {classify['critical_heads']}")
print(f"Backup heads: {classify['backup_heads']}")
print(f"Neutral heads: {classify['neutral_heads']}")

In [ ]:
curve = ablation_recovery_curve(model, tokens, metric_fn, ablate_heads=[(0,0), (0,1), (1,0)])
print(f"Recovery curve: {[round(v, 4) for v in curve['recovery_curve']]}")
print(f"Restoration order: {curve['restoration_order']}")